In [ ]:
import json

In [ ]:
def segment_to_ass(segment):
  line_start = format_time(segment["start"])
  line_end = format_time(segment["end"])
  
  ass_text = ""
  words = segment["words"]
  
  for i, w in enumerate(words):
    word_str = w["word"]
    duration = int(round((w["end"] - w["start"]) * 100))
    
    ass_text += f"{{\\kf{duration}}}{word_str}"
    
    if i < len(words) - 1:
      next_w = words[i+1]
      gap = int(round((next_w["start"] - w["end"]) * 100))
      
      if gap > 0:
        ass_text += f"{{\\kf{gap}}} "
      else:
        ass_text += "{\\kf0} "

  return f"Dialogue: 0,{line_start},{line_end},Karaoke,,0,0,0,,{ass_text}"

In [ ]:
def aligned_segments_to_ass(aligned_path):
  with open(aligned_path) as f:
    segments = json.load(f)
  ass = []

  header = """[Script Info]
  Title: Karaoke
  ScriptType: v4.00+

  [V4+ Styles]
  Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
  Style: Default,Arial,48,&H00FFFFFF,&H0000FFFF,&H00000000,&H64000000,0,0,1,2,0,2,10,10,10,1

  [Events]
  Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text

  """

  ass.append(header)

  for seg in segments:
    ass.append(segment_to_ass(seg))

  return "\n".join(ass)

In [ ]:
aligned_path = "output/aligned_segments.json"
with open(aligned_path) as f:
  segments = json.load(f)

ass = []

header = """[Script Info]
Title: Karaoke
ScriptType: v4.00+

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,48,&H00FFFFFF,&H0000FFFF,&H00000000,&H64000000,0,0,1,2,0,2,10,10,10,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text

"""

ass.append(header)

for seg in segments["segments"]:
  line_start = format_time(seg["start"])
  line_end = format_time(seg["end"])
  
  ass_text = ""
  words = seg["words"]
  
  for i, w in enumerate(words):
    word_str = w["word"]
    duration = int(round((w["end"] - w["start"]) * 100))
    
    ass_text += f"{{\\kf{duration}}}{word_str}"
    
    if i < len(words) - 1:
      next_w = words[i+1]
      gap = int(round((next_w["start"] - w["end"]) * 100))
      
      if gap > 0:
        ass_text += f"{{\\kf{gap}}} "
      else:
        ass_text += "{\\kf0} "

  ass.append(f"Dialogue: 0,{line_start},{line_end},Karaoke,,0,0,0,,{ass_text}")

result = "\n".join(ass)

type(result)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

def draw_centered(draw, text, font, y, image_width, font_color="white"):
  bbox = draw.textbbox((0, 0), text, font=font)
  width = bbox[2] - bbox[0]

  x = (image_width - width) // 2

  draw.text(
    (x, y),
    text,
    font=font,
    fill=font_color,
    stroke_width=8,
    stroke_fill="black"
  )

def create_thumbnail(song, artist):
  output_dir = Path("output/temp/thumbnail")
  output_dir.mkdir(parents=True, exist_ok=True)

  img = Image.open("data/assets/thumbnail.png")
  draw = ImageDraw.Draw(img)

  font_artist = ImageFont.truetype(
    "data/assets/fonts/Sweet Cucumber Mocktail.ttf", 250
  )
  font_song = ImageFont.truetype(
    "data/assets/fonts/Sweet Cucumber Mocktail.ttf", 200
  )
  font_subtitle = ImageFont.truetype(
    "data/assets/fonts/Sweet Cucumber Mocktail.ttf", 75
  )

  draw_centered(draw, song, font_song, 150, img.width, font_color="#7abaff")
  draw_centered(draw, artist, font_artist, 325, img.width)
  draw_centered(draw, "Karaoke Version", font_subtitle, 600, img.width, font_color="#8a8a8a")

  img.save(output_dir / "thumbnail.png", quality=100)

create_thumbnail("Wallows - 1980s Horror Film II (Karaoke)")